## Nội dung

- Feature engineering
- Phân cụm: KMeans, DBSCAN
- Kết hợp phân cụm và phân tích mô tả
- Kết hợp PCA và phân cụm
- Tập dữ liệu:
   1. Customer Personality Analysis (nguồn: https://www.kaggle.com/datasets/imakash3011/customer-personality-analysis)
   2. Taxi geolocation (nguồn: https://www.coursera.org/projects/clustering-geolocation-data-intelligently-python)

#### Cài đặt thư viện

In [ ]:
!pip install folium
!pip install kneed
!pip install yellowbrick
!pip install kneed

### A. Customer Personality Analysis

#### A1. Đọc dữ liệu

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
pd.set_option('display.max_columns', None)
customer_data = pd.read_csv('data/marketing_campaign.csv',
                           delimiter='\t', index_col='ID')
customer_data.head()

#### A2. Làm sạch
- Làm sạch dữ liệu.
- Trích xuất đặc trưng (đọc thêm: https://machinelearningcoban.com/general/2017/02/06/featureengineering/)

In [ ]:
customer_data.info()

Dữ liệu khá sạch và chỉ có 1 số lượng nhỏ income bị thiếu dữ liệu, có thể xử lý bằng cách thêm vào mean. Mã hoá 1 số đặc trưng kiểu loại.

In [ ]:
# Điền Income bị thiếu bằng giá trị trung bình
customer_data['Income'] = customer_data['Income'].fillna(customer_data['Income'].mean())
print('Số Income còn thiếu:', customer_data['Income'].isna().sum())

#### A3. Feature engineering

In [ ]:
import datetime

Đầu tiên là dữ liệu liên quan ngày tháng.

- `Year_Birth` không phải là đặc trưng lý tưởng cho bài toán này, ta sẽ chuyển nó sang thành tuổi `Age` (tính từ hiện tại)

In [ ]:
# Dùng năm hiện tại để tính tuổi
customer_data['Age'] = datetime.datetime.now().year - customer_data['Year_Birth']

- `Days_Since_Customer`, tương tự như `Year_Birth`, nên được chuyển thành số ngày đã tham gia `Enroll_days`.

In [ ]:
# Số ngày từ ngày đăng ký đến hiện tại
customer_data['Dt_Customer'] = pd.to_datetime(customer_data['Dt_Customer'], dayfirst=True)
now = pd.Timestamp.now().normalize()
customer_data['Enroll_days'] = (now - customer_data['Dt_Customer']).dt.days

- Các đặc trưng `Marital_Status`, `Kidhome`, `Teenhome` có thể được gom lại 1 thuộc tính duy nhất --> cô động hơn

In [ ]:
# Quy đổi tình trạng hôn nhân thành số người lớn, sau đó cộng trẻ em và thanh thiếu niên
customer_data['Marital_Status'] = customer_data['Marital_Status'].map(marital_map).fillna(1)
customer_data['Fam_Size'] = customer_data['Marital_Status'] + customer_data['Kidhome'] + customer_data['Teenhome']

- Gom các đặc trưng `AcceptedCmpX` thành 1 đặc trưng `Num_Accepted` (tổng)
- Làm tương tự cho các đặc trưng MntX, gom thành đặc trưng `MntTotal`

In [ ]:
# Tổng số chiến dịch trước đây mà khách hàng chấp nhận
cmp_cols = ['AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5']
customer_data['Num_Accepted'] = customer_data[cmp_cols].sum(axis=1)

In [ ]:
# Tổng tiền chi cho tất cả nhóm sản phẩm trong 2 năm
mnt_cols = ['MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
customer_data['MntTotal'] = customer_data[mnt_cols].sum(axis=1)

#### A4. Phân tích EDA

In [ ]:
customer_data.describe()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [ ]:
# Pair plot để quan sát quan hệ giữa các đặc trưng chính
sns.pairplot(customer_data[plot_cols].dropna(), diag_kind='hist')
plt.show()

In [ ]:
# Kiểm tra ngoại lệ bằng boxplot; giữ dữ liệu gốc và chỉ hiển thị để đánh giá
customer_data[plot_cols].plot(kind='box', subplots=True, layout=(1, len(plot_cols)),
                              figsize=(16, 4), sharex=False, sharey=False)
plt.tight_layout()
plt.show()
print('Số dòng:', len(customer_data), '| Income thiếu:', customer_data['Income'].isna().sum())

In [ ]:
import copy
df = customer_data.copy()

# Drop các cột không quan trọng
df.drop([
    'Dt_Customer', 'Year_Birth',
    'Kidhome', 'Teenhome', 'Marital_Status',
    'Z_CostContact', 'Z_Revenue',
    'MntWines', 'MntFruits', 'MntMeatProducts',
    'MntFishProducts', 'MntSweetProducts', 'MntGoldProds',
    'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3',
    'AcceptedCmp4','AcceptedCmp5',
    'Complain', 'Response', 'Education', 'Num_Accepted'
], axis=1, inplace=True)

#### A5. Tiền xử lý

**Mã hoá đặc trưng kiểu loại**

In [ ]:
# Mã hóa các biến phân loại còn lại thành số
from sklearn.preprocessing import LabelEncoder, StandardScaler
for col in df.select_dtypes(include=['object', 'category']).columns:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))
df.head()

**Chuẩn hoá các thuộc tính kiểu số**

In [ ]:
# Chuẩn hóa tất cả đặc trưng số để các cột có thang đo tương đương
scaler = StandardScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)
df_scaled.head()

In [ ]:
df.describe() # --> Kiểm tra mean và std có gần với giá trị (0, 1)?

#### A6. Phân cụm KMeans

In [ ]:
from sklearn.cluster import KMeans

Phương pháp Elbow: ước lượng số cụm

In [ ]:
# Elbow: thử K từ 1 đến 10
inertias = []
ks = range(1, 11)
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=99)
    km.fit(df_scaled)
    inertias.append(km.inertia_)
plt.figure(figsize=(8, 5))
plt.plot(list(ks), inertias, marker='o')
plt.xlabel('Số cụm K')
plt.ylabel('Inertia')
plt.title('Elbow method - Customer Personality')
plt.grid(True)
plt.show()

Sử dụng thư viện yellowbrick

In [ ]:
from yellowbrick.cluster import KElbowVisualizer

In [ ]:
# Yellowbrick hỗ trợ tìm vị trí khuỷu tay
model = KMeans(n_init=10, random_state=99)
visualizer = KElbowVisualizer(model, k=(2, 11), timings=False)
visualizer.fit(df_scaled)
visualizer.show()

**Phân cụm từ kết quả của phương pháp Elbow**

In [ ]:
# Chọn K=4 làm điểm bắt đầu; có thể điều chỉnh theo biểu đồ Elbow
K = 4
kmean = KMeans(n_clusters=K, n_init=20, random_state=99)
kmean.fit(df_scaled)
print('K đã chọn:', K)
print('Số lượng từng cụm:', pd.Series(kmean.labels_).value_counts().sort_index().to_dict())

**Sử dụng PCA giảm số chiều dữ liệu --> trực quan hoá kết quả phân cụm**

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
# Giảm dữ liệu chuẩn hóa xuống 2 chiều để vẽ
pca = PCA(n_components=2, random_state=99)
pca_scores = pca.fit_transform(df_scaled)
print('Tỷ lệ phương sai giải thích:', pca.explained_variance_ratio_)
print('Tổng phương sai giữ lại:', round(pca.explained_variance_ratio_.sum(), 3))

**Trực quan hoá kết quả phân cụm**

In [ ]:
# Vẽ các điểm trong không gian PCA và tô màu theo cụm
fig1, axes = plt.subplots(figsize=(8, 8))
sns.scatterplot(x=scores[:, 0], y=scores[:, 1], hue=labels, ax=axes,
                palette='Set1', legend='full', s=25)
axes.set_title('KMeans clusters in PCA 2D')
axes.set_xlabel('PC1')
axes.set_ylabel('PC2')
plt.show()

#### Phân tích kết quả phân cụm (KMeans)

In [ ]:
# Gán kết quả phân cụm vào df ban đầu ~ customer_data
kmean_df = customer_data.copy()
kmean_df['Cluster'] = kmean.labels_
kmean_df.head()

In [ ]:
def visualize_cluster_distribution(labels):
    '''
    Args:
        labels: kết quả phân cụm (danh sách nhãn)
    '''
    ulabels, count = np.unique(labels, return_counts=True)
    plt.figure(figsize=(6,4))

    colors = sns.color_palette('Set2')[0:5]
    
    labels = [f'Cluster {i+1}' for i in range(len(ulabels))]
    plt.pie(count, 
            explode=[0.05]*len(ulabels), 
            labels=labels,
            colors=colors,
            autopct='%1.1f%%',
            textprops=dict(color ="white", fontsize=10),
            counterclock=False,
            startangle=180,
            wedgeprops={"edgecolor":"gray",'linewidth': 1}
        )

    plt.axis('equal')
    plt.legend()
    plt.show()

In [ ]:
visualize_cluster_distribution(kmean.labels_)

**Phân tích đa biến + kết quả phân cụm**

In [ ]:
# So sánh thu nhập và tổng chi tiêu theo cụm
sns.scatterplot(data=kmean_df, x='Income', y='MntTotal', hue='Cluster', palette='Set1')
plt.title('Income và MntTotal theo cụm')
plt.show()

In [ ]:
# So sánh quy mô gia đình và tổng chi tiêu theo cụm
sns.scatterplot(data=kmean_df, x='Fam_Size', y='MntTotal', hue='Cluster', palette='Set1')
plt.title('Fam_Size và MntTotal theo cụm')
plt.show()

**Phân tích danh mục mua hàng**

In [ ]:
# Tóm tắt đặc điểm trung bình của các cụm và mức chi tiêu theo danh mục
print(kmean_df.groupby('Cluster')[['Age', 'Income', 'Fam_Size', 'MntTotal', 'NumDealsPurchases',
                                  'NumWebPurchases', 'NumStorePurchases', 'NumWebVisitsMonth']].mean().round(2))
spending_by_cluster = kmean_df.groupby('Cluster')[mnt_cols].mean().round(2)
display(spending_by_cluster)
spending_by_cluster.T.plot(kind='bar', figsize=(12, 5))
plt.ylabel('Chi tiêu trung bình')
plt.title('Chi tiêu từng danh mục theo cụm')
plt.tight_layout()
plt.show()

#### A7. Kết hợp PCA và phân cụm
- Biến đổi PCA với số lượng component phù hợp (giữ lại 70-80% thông tin ban đầu).
- Ước lượng số cụm dùng elbow (dùng dữ liệu pca).
- Với số cụm phù hợp, thực hiện phân cụm trên dữ liệu pca.
- Vẽ phân bố kết quả phân cụm.
- Vẽ biểu đồ scatter MntTotal-Income-Cluster.

In [ ]:
# vẽ biểu đồ sreeplot
def scree_plot(pca_model, xlabels):
    fig, ax = plt.subplots(figsize=(12, 8))
    
    # explained variance ratio của mô hình pca
    expl_var_ratio = pca_model.explained_variance_ratio_
    # tính cumulative sum của expl_var_ratio (numpy.cumsum)
    expl_var_cumsum = np.cumsum(expl_var_ratio)
    xtick_indx = range(1, len(expl_var_ratio) + 1)
    
    # vẽ các bar thể hiện explained_var
    ax.bar(xtick_indx, expl_var_ratio)
    ax.plot(xtick_indx, expl_var_cumsum, marker='o', color='k')
    
    # set label
    ax.set_xticks(xtick_indx)
    ax.set_xticklabels(xlabels)
    ax.set_xlabel('Principle Components')
    ax.set_ylabel('% variance')
    ax.set_title('Scree Plot')
    
    plt.grid(True)

In [ ]:
scree_plot(pca, [f'pc{i+1}' for i in range(len(df.columns))])

In [ ]:
# PCA 4 chiều để giữ lại nhiều thông tin hơn cho bước phân cụm
pca_4d = PCA(n_components=4, random_state=99)
pca_transformed_4d = pca_4d.fit_transform(df_scaled)
print('Tổng phương sai giữ lại với 4 PC:', round(pca_4d.explained_variance_ratio_.sum(), 3))

In [ ]:
model = KMeans(n_init=20, random_state=99)
visualizer = KElbowVisualizer(model, k=10)
visualizer.fit(pca_transformed_4d)
visualizer.show()

In [ ]:
# Tạo phân cụm riêng trên PCA 4 chiều (không dùng nhãn của KMeans trước đó)
kmeans_pca = KMeans(n_clusters=4, n_init=20, random_state=99)
kmean_pca_df = customer_data.copy()
kmean_pca_df['Cluster'] = kmeans_pca.fit_predict(pca_transformed_4d)
kmean_pca_df.head()

In [ ]:
# Dùng nhãn KMeans từ PCA để xem thu nhập và chi tiêu
sns.scatterplot(data=kmean_pca_df, x='Income', y='MntTotal', hue='Cluster', palette='Set1')
plt.title('Income và MntTotal theo cụm (PCA)')
plt.show()

#### A9. Phân tích kết quả phân cụm (DBSCAN)

In [ ]:
from sklearn.cluster import DBSCAN

**Sử dụng knee plot để ước lượng eps**

In [ ]:
len(df.columns)

In [ ]:
from sklearn.neighbors import NearestNeighbors
def knee_plot(n_neighs, data):
    neigh = NearestNeighbors(n_neighbors=n_neighs)
    nbrs = neigh.fit(data)
    distances, indices = nbrs.kneighbors(data)
    distances = np.sort(distances[:, -1])
    plt.plot(distances)
    plt.xlabel('Điểm, đã sắp theo khoảng cách')
    plt.ylabel(f'Khoảng cách tới láng giềng thứ {n_neighs}')
    plt.grid(True)
    return distances

**Sử dụng PCA thu giảm số chiều --> phân cụm**

**Chia nhỏ đặc trưng**

In [ ]:
sub_df = df[['Income', 'MntTotal', 'Enroll_days']]

### B. Dữ liệu taxi 

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('data/taxi_data.csv')
df.head()

**Biểu diễn giá trị toạ độ trên bản đồ**

In [ ]:
import folium, re

In [ ]:
X = df[['LAT','LON']].values
m = folium.Map(location=[X[:,0].mean(), X[:,1].mean()], tiles='OpenStreetMap', zoom_start=10)

In [ ]:
for _,row in df.iterrows():
    folium.CircleMarker(
        location=[row.LAT,row.LON],
        radius=5,
        popup=re.sub(r'\W+',' ',row.NAME),
        fill=True
    ).add_to(m)

# Loại bỏ non-word chars
title = '''<h3 align="center" style="font-size:30px"><b>Given Data</b></h3>'''
m.get_root().html.add_child(folium.Element(title))
m

In [ ]:
# Knee plot trên tọa độ đã chuẩn hóa
from sklearn.preprocessing import StandardScaler
taxi_scaler = StandardScaler()
X_scaled = taxi_scaler.fit_transform(df[['LAT', 'LON']].values)
distances = knee_plot(5, X_scaled)
plt.figure(figsize=(8, 4))
plt.plot(np.sort(distances))
plt.xlabel('Điểm (đã sắp theo khoảng cách)')
plt.ylabel('Khoảng cách k-láng giềng')
plt.title('K-distance plot cho taxi')
plt.grid(True)
plt.show()

In [ ]:
#First create an array of color schemes
colors = ['royalblue', 'maroon', 'forestgreen', 'mediumorchid', 'tan', 'deeppink', 
          'olive', 'goldenrod', 'lightcyan', 'navy','springgreen','midnightblue',
         'red','brown','limegreen','lime','pink','orchid','crimson','m']*10

In [ ]:
# DBSCAN trên tọa độ địa lý đã chuẩn hóa; eps có thể tinh chỉnh dựa vào knee plot
from sklearn.cluster import DBSCAN
eps = 0.12
min_samples = 5
taxi_labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_scaled)
df['Cluster'] = taxi_labels
print('eps:', eps, '| min_samples:', min_samples)
print('Số cụm (không tính noise):', len(set(taxi_labels) - {-1}))
print('Số điểm noise:', int((taxi_labels == -1).sum()))
print(pd.Series(taxi_labels).value_counts().sort_index())

In [ ]:
def create_map(cluster_column, colors_column, title):
    m = folium.Map(location=[X[:, 0].mean(), X[:, 1].mean()],
                   tiles='OpenStreetMap', zoom_start=10)
    for _, row in df.iterrows():
        folium.CircleMarker(
            location=[row.LAT, row.LON],
            radius=5,
            popup=str(row[cluster_column]),
            fill=True,
            color=row[colors_column],
            fill_color=row[colors_column],
            fill_opacity=0.8
        ).add_to(m)
    print(title)
    return m

# Gán màu cho cụm DBSCAN, điểm nhiễu màu đen
unique_labels = sorted(df['Cluster'].unique())
palette = ['royalblue', 'maroon', 'forestgreen', 'mediumorchid', 'tan',
           'deeppink', 'olive', 'goldenrod', 'navy', 'red', 'brown']
color_map = {-1: 'black'}
for i, label in enumerate([x for x in unique_labels if x != -1]):
    color_map[label] = palette[i % len(palette)]
df['Color'] = df['Cluster'].map(color_map)
create_map('Cluster', 'Color', 'Taxi ranks clustered with DBSCAN')